# The corpus against Tatoeba: a methodological comparison

**Question.** Tatoeba is the standard test of cross-lingual sentence retrieval. What does a
verse-aligned, meaning-controlled corpus show that Tatoeba cannot?

**Design.** Run the same four encoders, with the same protocol, on both benchmarks for every
language they share, then compare on four things decided in advance:

| | Question | Measured by |
|---|---|---|
| **A** | Do the two benchmarks order languages the same way? | Spearman ρ over shared languages, per encoder |
| **B** | Which benchmark can tell encoders apart? | share of languages at ceiling; spread across languages; per language, how many of the 6 encoder pairs a paired bootstrap separates |
| **C** | Is Tatoeba's language ordering confounded by content? | Tatoeba accuracy against the length and size of each language's test set; English-side overlap between languages' test sets |
| **D** | What can only the corpus measure? | languages absent from Tatoeba, non-English pairs, units per language, identical content |

**Protocol, identical on both benchmarks.** English-paired retrieval in both directions (X→English and
English→X), top-1 accuracy by cosine similarity over normalized embeddings, averaged over the two
directions. The corpus side uses the paper's 1,000-verse sample (seed 0), so its numbers reproduce the
English rows of `results/retrieval_pairs.csv`; the notebook checks this. Tatoeba test sets are the
standard LASER release (up to 1,000 pairs per language). Because a smaller candidate pool is easier,
the corpus is also scored on a random subsample matched to each language's Tatoeba size.

**Languages.** 27 corpus languages are in Tatoeba. Kurdish is excluded from the main analysis
because Tatoeba's Kurdish is Kurmanji in Latin script and the corpus's is Sorani in Arabic script.
Serbian and Uzbek mix Latin and Cyrillic in Tatoeba; results are reported with and without them.
The 12 corpus languages Tatoeba lacks are Asante Twi, Assamese, Gujarati, Hausa, Kannada, Kyrgyz,
Lingala, Mooré, Oromo, Pashto, Tajik and Yoruba.

**Runtime.** GPU (T4), about 20 minutes. The setup cell asks for four files **one at a time**, and
says where each one is: `translations.zip`, `retrieval_pairs.csv`, `retrieval_per_language.csv` and
`joint_language_table.csv`. Tatoeba downloads from the LASER repository.

## 0. Setup

In [ ]:
!pip -q install sentence-transformers
import os, io, csv, glob, json, zipfile, itertools, unicodedata, urllib.request, time
from collections import Counter
import numpy as np, pandas as pd
from scipy import stats
import torch

OUT = "results_tatoeba"; os.makedirs(OUT, exist_ok=True)
N_BOOT = 1000
ENCODERS = {"labse": "sentence-transformers/LaBSE", "bgem3": "BAAI/bge-m3",
            "e5": "intfloat/multilingual-e5-base",
            "mpnet": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"}
# Each input is requested on its own, so the files can come from different folders.
NEED = {
    "translations.zip":           "data/",
    "retrieval_pairs.csv":        "results/",
    "retrieval_per_language.csv": "results/",
    "joint_language_table.csv":   "data/metadata/",
}

def ask_for(name, where):
    # open an upload dialog for one file; accept it even if the browser renamed it
    from google.colab import files
    while not os.path.exists(name):
        print(f"\nUpload  {name}   (found in: {where})")
        up = files.upload()
        if not up:
            print("  nothing uploaded, asking again"); continue
        got = list(up)[0]
        stem, ext = os.path.splitext(name)
        if got != name and got.startswith(stem) and got.endswith(ext):   # e.g. "retrieval_pairs (1).csv"
            os.replace(got, name)
        elif got != name:
            print(f"  got {got}, expected {name}; ignoring it and asking again")
            os.remove(got); continue
        print(f"  ok: {name} ({os.path.getsize(name):,} bytes)")

try:
    import google.colab  # noqa: F401
    for name, where in NEED.items():
        if os.path.exists(name):
            print(f"already here: {name}")
        else:
            ask_for(name, where)
except ImportError:
    pass
for f in NEED:
    assert os.path.exists(f), f"missing {f}"
GPU = torch.cuda.is_available()
print("GPU:", GPU)
if not GPU:
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4.")

## 1. Corpus and the paper's 1,000-verse sample

Reader, naming and sampling are copied from `02_crosslingual_retrieval.ipynb`, so the verses are the
same ones the paper's retrieval numbers were computed on.

In [ ]:
if not os.path.exists("corpus"):
    with zipfile.ZipFile("translations.zip") as z:
        z.extractall("corpus")
csv_files = sorted(glob.glob("corpus/**/*.csv", recursive=True))

def load_csv(fp):
    keys = {}
    with open(fp, encoding="utf-8", errors="replace") as f:
        rdr = csv.reader(f); started = False
        for row in rdr:
            if not started:
                low = [(x or "").strip().lower() for x in row]
                if "sura" in low and "aya" in low:
                    started = True; iS = low.index("sura"); iA = low.index("aya")
                    iT = low.index("translation") if "translation" in low else 3
                continue
            try: keys[(int(row[iS]), int(row[iA]))] = row[iT]
            except Exception: pass
    return keys

lang = lambda fp: os.path.basename(fp).split("_")[0]
cnt = Counter(lang(f) for f in csv_files)
disp = {f: (lang(f) if cnt[lang(f)] == 1 else lang(f) + "-" + os.path.basename(f).split("_")[1]) for f in csv_files}
TEXT = {disp[f]: load_csv(f) for f in csv_files}
common = sorted(set.intersection(*[set(v) for v in TEXT.values()]))
assert len(TEXT) == 41 and len(common) == 6236
rng0 = np.random.default_rng(0)
SAMPLE = [common[i] for i in rng0.choice(len(common), size=1000, replace=False)]   # as in notebook 02
print(len(TEXT), "files,", len(SAMPLE), "sampled verses")

# corpus language -> Tatoeba (LASER) code; one Bosnian translation, as in the paper's n = 40 analyses
TATOEBA = {"albanian": "sqi", "azeri": "aze", "bosnian-rwwad": "bos", "chinese": "cmn", "croatian": "hrv",
           "dutch": "nld", "french": "fra", "german": "deu", "indonesian": "ind", "japanese": "jpn",
           "khmer": "khm", "kurdish": "kur", "lithuanian": "lit", "macedonian": "mkd", "malayalam": "mal",
           "persian": "pes", "portuguese": "por", "romanian": "ron", "serbian": "srp", "spanish": "spa",
           "tagalog": "tgl", "tamil": "tam", "turkish": "tur", "urdu": "urd", "uyghur": "uig",
           "uzbek": "uzb", "vietnamese": "vie"}
SCRIPT_MISMATCH = {"kurdish"}          # Kurmanji/Latin in Tatoeba vs Sorani/Arabic in the corpus
MIXED_SCRIPT = {"serbian", "uzbek"}    # Tatoeba mixes Latin and Cyrillic
MAIN = [l for l in TATOEBA if l not in SCRIPT_MISMATCH]
NOT_IN_TATOEBA = sorted(l for l in TEXT if l not in TATOEBA and l not in ("english", "bosnian-mihanovich"))
print(f"{len(TATOEBA)} shared languages ({len(MAIN)} in the main analysis); "
      f"{len(NOT_IN_TATOEBA)} corpus languages not in Tatoeba: {NOT_IN_TATOEBA}")

## 2. Tatoeba, and what its test sets contain (no models)

This section answers **C** and part of **D** before any encoder is run.

In [ ]:
BASE = "https://raw.githubusercontent.com/facebookresearch/LASER/main/data/tatoeba/v1"
os.makedirs("tatoeba", exist_ok=True)
TAT = {}
for l, c in TATOEBA.items():
    pair = {}
    for side in (c, "eng"):
        fp = f"tatoeba/tatoeba.{c}-eng.{side}"
        if not os.path.exists(fp):
            urllib.request.urlretrieve(f"{BASE}/tatoeba.{c}-eng.{side}", fp)
        pair[side] = [s.rstrip("\n") for s in open(fp, encoding="utf-8")]
    assert len(pair[c]) == len(pair["eng"]), (l, len(pair[c]), len(pair["eng"]))
    TAT[l] = {"x": pair[c], "eng": pair["eng"]}

def main_script(ts):
    k = Counter()
    for t in ts[:300]:
        for ch in t:
            if ch.isalpha(): k[unicodedata.name(ch, "?").split()[0]] += 1
    tot = sum(k.values()) or 1
    return ", ".join(f"{s} {n/tot:.0%}" for s, n in k.most_common(2))

eng_words_corpus = np.mean([len(str(TEXT["english"][k]).split()) for k in common])
rows = []
for l in TATOEBA:
    e = TAT[l]["eng"]
    rows.append({"language": l, "code": TATOEBA[l], "tatoeba_pairs": len(e),
                 "tatoeba_eng_words_per_sentence": round(np.mean([len(s.split()) for s in e]), 2),
                 "tatoeba_script": main_script(TAT[l]["x"]),
                 "corpus_script": main_script([str(TEXT[l][k]) for k in common[:300]])})
CONTENT = pd.DataFrame(rows).set_index("language")
CONTENT.to_csv(f"{OUT}/tatoeba_content.csv")
print(CONTENT.to_string())

S = {l: set(TAT[l]["eng"]) for l in TATOEBA}
jac = [len(S[a] & S[b]) / len(S[a] | S[b]) for a, b in itertools.combinations(TATOEBA, 2)]
shared_all = set.intersection(*S.values())
CONTENT_SUMMARY = {
    "n_shared_languages": len(TATOEBA), "n_main": len(MAIN),
    "tatoeba_pairs_min": int(CONTENT.tatoeba_pairs.min()), "tatoeba_pairs_median": float(CONTENT.tatoeba_pairs.median()),
    "tatoeba_eng_words_median": float(CONTENT.tatoeba_eng_words_per_sentence.median()),
    "tatoeba_eng_words_range": [float(CONTENT.tatoeba_eng_words_per_sentence.min()),
                                float(CONTENT.tatoeba_eng_words_per_sentence.max())],
    "corpus_eng_words_per_verse": round(float(eng_words_corpus), 1),
    "english_overlap_median_jaccard": round(float(np.median(jac)), 4),
    "english_overlap_max_jaccard": round(float(np.max(jac)), 3),
    "english_sentences_shared_by_all": len(shared_all),
    "distinct_english_sentences": len(set().union(*S.values())),
}
print(json.dumps(CONTENT_SUMMARY, indent=1))
print("\nOn the corpus every language renders the same 6,236 verses; on Tatoeba the English sides of two "
      "languages' test sets share a median Jaccard of", CONTENT_SUMMARY["english_overlap_median_jaccard"])

## 3. Embed and score

For every encoder: the corpus sample in English and the 27 shared languages, and each Tatoeba test
set. Per-item correctness is kept for both directions so that Section 5 can bootstrap over items.

In [ ]:
from sentence_transformers import SentenceTransformer

def correct(A, B):
    # per-item top-1 correctness, A->B and B->A, for aligned row-normalized embeddings
    S = A @ B.T
    n = len(A)
    return (S.argmax(1) == np.arange(n)), (S.argmax(0) == np.arange(n))

rng_size = np.random.default_rng(42)
SIZE_IDX = {l: np.sort(rng_size.choice(1000, size=min(1000, CONTENT.loc[l, "tatoeba_pairs"]), replace=False))
            for l in TATOEBA}

ITEMS, rows = {}, []
t0 = time.time()
for tag, name in ENCODERS.items():
    model = SentenceTransformer(name, device="cuda" if GPU else "cpu")
    pre = "query: " if "e5" in name else ""
    emb = lambda ts: model.encode([pre + t for t in ts], batch_size=64, convert_to_numpy=True,
                                  normalize_embeddings=True, show_progress_bar=False)
    E_corpus_en = emb([str(TEXT["english"][k]) for k in SAMPLE])
    for l in TATOEBA:
        X = emb([str(TEXT[l][k]) for k in SAMPLE])
        c1, c2 = correct(X, E_corpus_en)                       # X->eng, eng->X
        i = SIZE_IDX[l]; s1, s2 = correct(X[i], E_corpus_en[i])
        T1, T2 = correct(emb(TAT[l]["x"]), emb(TAT[l]["eng"]))
        ITEMS[(tag, l, "corpus")] = np.concatenate([c1, c2])
        ITEMS[(tag, l, "tatoeba")] = np.concatenate([T1, T2])
        rows.append({"encoder": tag, "language": l,
                     "corpus_x2en": c1.mean(), "corpus_en2x": c2.mean(), "corpus": (c1.mean() + c2.mean()) / 2,
                     "corpus_size_matched": (s1.mean() + s2.mean()) / 2,
                     "tatoeba_x2en": T1.mean(), "tatoeba_en2x": T2.mean(), "tatoeba": (T1.mean() + T2.mean()) / 2,
                     "tatoeba_pairs": len(T1)})
    print(f"  {tag:6s} done ({time.time()-t0:.0f}s)", flush=True)
    del model
    if GPU: torch.cuda.empty_cache()
ACC = pd.DataFrame(rows)
ACC.to_csv(f"{OUT}/accuracy_by_language.csv", index=False)
np.savez_compressed(f"{OUT}/per_item_correct.npz", **{"|".join(k): v for k, v in ITEMS.items()})
print(ACC.groupby("encoder")[["corpus", "corpus_size_matched", "tatoeba"]].mean().round(3))

**Protocol check.** The corpus scores must reproduce the English rows of the paper's
`retrieval_pairs.csv`. Small differences (≤ 0.005) can come from library versions or GPU arithmetic;
larger ones mean the protocol differs and nothing below is comparable.

In [ ]:
rp = pd.read_csv("retrieval_pairs.csv")
TAG = {v: k for k, v in ENCODERS.items()}
rp["encoder"] = rp.model.map(TAG)
chk = []
for _, r in ACC.iterrows():
    a = rp[(rp.encoder == r.encoder) & (rp.src == r.language) & (rp.tgt == "english")].top1
    b = rp[(rp.encoder == r.encoder) & (rp.src == "english") & (rp.tgt == r.language)].top1
    if len(a) and len(b):
        chk.append({"encoder": r.encoder, "language": r.language,
                    "diff_x2en": r.corpus_x2en - a.iloc[0], "diff_en2x": r.corpus_en2x - b.iloc[0]})
CHK = pd.DataFrame(chk)
worst = CHK[["diff_x2en", "diff_en2x"]].abs().max().max()
print(f"max |difference| from retrieval_pairs.csv over {len(CHK)} language-encoder cells: {worst:.4f}")
print("PROTOCOL REPRODUCES" if worst <= 0.005 else "WARNING: protocol does not reproduce; inspect CHK")

## 4. A — do the benchmarks order languages the same way?

In [ ]:
pl = pd.read_csv("retrieval_per_language.csv"); pl["encoder"] = pl.model.map(TAG)
rows = []
for enc in ENCODERS:
    for label, langs in [("main (26)", MAIN), ("without Serbian, Uzbek", [l for l in MAIN if l not in MIXED_SCRIPT])]:
        d = ACC[(ACC.encoder == enc) & ACC.language.isin(langs)].set_index("language")
        allpairs = pl[pl.encoder == enc].set_index("language").loc[d.index, "mean_top1"]
        r1, p1 = stats.spearmanr(d.tatoeba, d.corpus)
        r2, p2 = stats.spearmanr(d.tatoeba, d.corpus_size_matched)
        r3, p3 = stats.spearmanr(d.tatoeba, allpairs)
        rows.append({"encoder": enc, "languages": label, "n": len(d),
                     "rho_eng_paired": round(r1, 3), "p": round(p1, 4),
                     "rho_size_matched": round(r2, 3), "rho_vs_all_pairs": round(r3, 3)})
AGREE = pd.DataFrame(rows)
AGREE.to_csv(f"{OUT}/A_rank_agreement.csv", index=False)
print(AGREE.to_string(index=False))

## 5. B — which benchmark can tell encoders apart?

For each benchmark: how many languages sit at ceiling (≥ 0.95), how widely languages spread, and, per
language, how many of the six encoder pairs a paired bootstrap over items separates (95% interval of
the accuracy difference excludes zero). A benchmark at ceiling cannot rank encoders, however many
languages it covers.

In [ ]:
rng_b = np.random.default_rng(7)
pairs = list(itertools.combinations(ENCODERS, 2))
rows = []
for l in MAIN:
    for bench in ["corpus", "tatoeba"]:
        v = {e: ITEMS[(e, l, bench)].astype(float) for e in ENCODERS}
        n = len(v["labse"]); idx = rng_b.integers(0, n, size=(N_BOOT, n))
        sep = 0
        for a, b in pairs:
            d = (v[a] - v[b])[idx].mean(1)
            lo, hi = np.percentile(d, [2.5, 97.5])
            sep += int(lo > 0 or hi < 0)
        rows.append({"language": l, "benchmark": bench, "encoder_pairs_separated": sep})
SEP = pd.DataFrame(rows).pivot(index="language", columns="benchmark", values="encoder_pairs_separated")
SEP.to_csv(f"{OUT}/B_encoder_pairs_separated.csv")

rows = []
for enc in ENCODERS:
    d = ACC[(ACC.encoder == enc) & ACC.language.isin(MAIN)]
    for bench in ["corpus", "corpus_size_matched", "tatoeba"]:
        x = d[bench].values
        rows.append({"encoder": enc, "benchmark": bench, "mean": round(x.mean(), 3),
                     "share_at_ceiling_0.95": round(float((x >= 0.95).mean()), 3),
                     "sd": round(x.std(ddof=1), 3), "iqr": round(float(np.subtract(*np.percentile(x, [75, 25]))), 3),
                     "min": round(x.min(), 3), "max": round(x.max(), 3)})
SPREAD = pd.DataFrame(rows)
SPREAD.to_csv(f"{OUT}/B_ceiling_and_spread.csv", index=False)
print(SPREAD.to_string(index=False))
print(f"\nEncoder pairs separated per language (of 6): corpus median {SEP.corpus.median():.1f}, "
      f"Tatoeba median {SEP.tatoeba.median():.1f}")
print(f"languages where the corpus separates more pairs: {(SEP.corpus > SEP.tatoeba).sum()}, "
      f"fewer: {(SEP.corpus < SEP.tatoeba).sum()}, equal: {(SEP.corpus == SEP.tatoeba).sum()}")

rank = ACC[ACC.language.isin(MAIN)].groupby("encoder")[["corpus", "tatoeba"]].mean()
print("\nOverall encoder ranking:")
print(rank.rank(ascending=False).astype(int).join(rank.round(3), rsuffix="_mean").to_string())
taus = []
for l in MAIN:
    d = ACC[ACC.language == l].set_index("encoder")
    taus.append(stats.kendalltau(d.corpus, d.tatoeba)[0])
print(f"per-language Kendall tau between the two benchmarks' encoder orderings: median {np.nanmedian(taus):.2f}")

## 6. C — is Tatoeba's language ordering confounded by content?

Each Tatoeba language has its own sentences, so a language can score higher because its sentences
are shorter, fewer, or easier. The corpus rules this out by construction. Here we measure how much
Tatoeba accuracy moves with test-set properties that have nothing to do with the language.

In [ ]:
rows = []
for enc in ENCODERS:
    d = ACC[(ACC.encoder == enc) & ACC.language.isin(MAIN)].set_index("language").drop(columns="tatoeba_pairs").join(CONTENT)
    for prop in ["tatoeba_eng_words_per_sentence", "tatoeba_pairs"]:
        r, p = stats.spearmanr(d.tatoeba, d[prop])
        rows.append({"encoder": enc, "tatoeba_accuracy_vs": prop, "rho": round(r, 3), "p": round(p, 4)})
CONF = pd.DataFrame(rows)
CONF.to_csv(f"{OUT}/C_content_confound.csv", index=False)
print(CONF.to_string(index=False))
print("\nAt n = 26, |rho| below about 0.39 is indistinguishable from zero.")

## 7. D — resources side by side

In [ ]:
n_corpus_langs = len([l for l in TEXT if l != "bosnian-mihanovich"])
RES = pd.DataFrame([
    {"resource": "This corpus", "languages": f"{n_corpus_langs} (+ Arabic source)",
     "units_per_language": 6236, "same_content_in_every_language": "yes (verse-aligned)",
     "non_English_pairs_scorable": n_corpus_langs * (n_corpus_langs - 1),
     "English_words_per_unit": CONTENT_SUMMARY["corpus_eng_words_per_verse"], "domain": "religious, formal"},
    {"resource": "Tatoeba (LASER test sets), shared languages", "languages": f"{len(TATOEBA)} of ours",
     "units_per_language": f"{CONTENT_SUMMARY['tatoeba_pairs_min']}-1000",
     "same_content_in_every_language": f"no (median English-side Jaccard {CONTENT_SUMMARY['english_overlap_median_jaccard']})",
     "non_English_pairs_scorable": 0,
     "English_words_per_unit": CONTENT_SUMMARY["tatoeba_eng_words_median"], "domain": "short everyday sentences"},
])
RES.to_csv(f"{OUT}/D_resources.csv", index=False)
print(RES.T.to_string(header=False))
print(f"\nCorpus languages Tatoeba does not cover ({len(NOT_IN_TATOEBA)}):", ", ".join(NOT_IN_TATOEBA))

## Summary and download

In [ ]:
m = AGREE[AGREE.languages == "main (26)"]
S = {"content": CONTENT_SUMMARY,
     "protocol_check_max_abs_diff": round(float(worst), 4),
     "A_rank_agreement": m.set_index("encoder")[["n", "rho_eng_paired", "p", "rho_size_matched", "rho_vs_all_pairs"]].to_dict(orient="index"),
     "B_spread": SPREAD.set_index(["encoder", "benchmark"]).reset_index().to_dict(orient="records"),
     "B_pairs_separated_median": {"corpus": float(SEP.corpus.median()), "tatoeba": float(SEP.tatoeba.median())},
     "B_languages_corpus_separates_more": int((SEP.corpus > SEP.tatoeba).sum()),
     "B_languages_tatoeba_separates_more": int((SEP.tatoeba > SEP.corpus).sum()),
     "C_content_confound": CONF.to_dict(orient="records"),
     "D_not_in_tatoeba": NOT_IN_TATOEBA}
json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=float)
print(json.dumps(S, indent=1, default=float)[:5000])
import shutil; shutil.make_archive("results_tatoeba", "zip", OUT)
try:
    from google.colab import files; files.download("results_tatoeba.zip")
except Exception:
    print("written results_tatoeba.zip")

## Reading the results

Decided before running, and reported whichever way they come out.

| Result | Meaning |
|---|---|
| **A** ρ ≥ 0.7 for most encoders | The corpus orders shared languages as Tatoeba does: it measures the same thing, and extends it to 12 languages Tatoeba lacks. |
| **A** ρ well below 0.7 | The benchmarks disagree. Section C says whether Tatoeba's content differences could explain it; if not, domain is the likely cause and must be said. |
| **B** Tatoeba near ceiling for the strong encoders while the corpus spreads languages, and the corpus separates more encoder pairs | The corpus discriminates where Tatoeba cannot. |
| **B** Tatoeba separates as many or more pairs | Its sentence-level design already has enough resolution on these languages; the corpus's advantage is coverage and parallelism, not resolution. |
| **C** Tatoeba accuracy tracks test-set length or size | Part of Tatoeba's language ordering reflects which sentences each language received, which a meaning-controlled corpus rules out. |
| **D** | Not an experiment: coverage, parallel structure and scale, measured rather than asserted. |

The domain difference is real either way. Religious verse is longer and more formal than Tatoeba's
everyday sentences, so absolute accuracies are not comparable across the two; the comparisons above
are within benchmark (rankings, spread, resolution) for that reason.